# Scenario 2 — A Door and a Mobile Robot (Direct Invocation)

The **direct workflow/state invocation** pattern — *not* operation based. A door resource exposes open/close workflows and a live status StateProperty, with **no operation queue**; its workflow endpoints execute synchronously. A minimal mobile robot discovers the door purely through the knowledge graph (a SPARQL query), reads its live status over the StateProperty GET endpoint, and — finding it closed — invokes the door's open workflow directly at the endpoint it found in the graph.

In [1]:
import os
import threading
import time

import httpx
import uvicorn
from graph_db_interface import GraphDB
from rdflib.namespace import RDF

from handlers import door_close, door_open, door_status, reset_door
from kapps_ogm import OGM
from kapps_semantic_middleware import SemanticMiddleware
from kapps_semantic_middleware.registration import mint_state_property_iri, mint_workflow_iri
from kapps_semantic_middleware.vocabulary import SVC

import seed

db = GraphDB.from_env()
print(f"Connected to GraphDB repository: {os.getenv('GRAPHDB_REPOSITORY')}")

INFO:KafkaManager:KafkaManager initialized


INFO:GraphDB:Using GraphDB repository 'Tests' as user 'etienneh'.


Connected to GraphDB repository: Tests


## Step 1 — Seed a Clean Repository

Load the Scenario 2 ontology and create the door + mobile-robot resources.

In [2]:
reset_door()
seed.seed_scenario2(db)
print('Door resource: ', db.triple_exists((seed.DOOR_RESOURCE, RDF.type, seed.DOOR_RESOURCE_CLASS)))
print('Robot resource:', db.triple_exists((seed.MOBILE_ROBOT, RDF.type, seed.MOBILE_ROBOT_RESOURCE_CLASS)))

Door resource:  True
Robot resource: True


## Step 2 — Start the Door Middleware

Register the door's two workflows (open, close) and its live status StateProperty. This scenario never uses the operation queue — the workflow endpoints execute synchronously when invoked directly.

In [3]:
door = SemanticMiddleware(
    mode='resource', resource_iri=seed.DOOR_RESOURCE,
    service_class=seed.DOOR_SERVICE_CLASS, ogm=OGM(db=db),
    host='127.0.0.1', port=8997,
)
door.workflow(capability_class=seed.DOOR_OPEN_CAPABILITY_CLASS, workflow_class=seed.DOOR_OPEN_WORKFLOW_CLASS)(door_open)
door.workflow(capability_class=seed.DOOR_CLOSE_CAPABILITY_CLASS, workflow_class=seed.DOOR_CLOSE_WORKFLOW_CLASS)(door_close)
door.state(capability_class=seed.DOOR_STATUS_CAPABILITY_CLASS, state_property_class=seed.DOOR_STATUS_STATE_CLASS)(door_status)

# Uvicorn installs signal handlers only on the main thread. On this one, SIGTERM never
# reaches the ASGI lifespan, so the middleware's on_shutdown deregistration never runs.
# The shutdown cell at the end stops the server with should_exit + join, which runs that
# lifespan shutdown with no signal involved. Skip it and this notebook leaves an
# svc:address published for a kernel that is gone (#65).
config = uvicorn.Config(door.app, host='127.0.0.1', port=8997, log_level='warning')
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
t0 = time.time()
while not server.started and time.time() - t0 < 30:
    time.sleep(0.05)
print('Door middleware started on port 8997')

Door middleware started on port 8997


## Step 3 — Inspect What Registration Wrote

Both workflows and the state property are in the graph with reachable endpoints — these are the URLs the robot will discover.

In [4]:
# Per-instance since ADR 0022 — read off the instance rather than rebuilt from the resource.
service_iri = door.service_iri
open_wf = mint_workflow_iri(service_iri, 'door_open')
status_sp = mint_state_property_iri(service_iri, 'door_status')
assert db.triple_exists((open_wf, SVC.isWorkflowOf, service_iri))
assert db.triple_exists((status_sp, SVC.isStatePropertyOf, service_iri))
print('open workflow endpoint:', list(db.triples_get(sub=open_wf, pred=SVC.endpoint)))
print('status state endpoint: ', list(db.triples_get(sub=status_sp, pred=SVC.endpoint)))

open workflow endpoint: [(IRI('https://example.org/kapps-demo#door_042_service_workflow_door_open'), IRI('https://w3id.org/circularfactory/Service#endpoint'), 'http://127.0.0.1:8997/workflows/door_open/execute')]
status state endpoint:  [(IRI('https://example.org/kapps-demo#door_042_service_state_door_status'), IRI('https://w3id.org/circularfactory/Service#endpoint'), 'http://127.0.0.1:8997/state/door_status')]


## Step 4 — The Mobile Robot Discovers and Passes the Door

The robot's scripted behaviour: discover the door via a **SPARQL** query, read its live status over REST, and if closed invoke the open workflow **directly** at the execute URL found in the graph. Deterministic — the only forks are *is the door open?* and, if not, *open it*. On return the door auto-closes after 30 s, but the drop-off took less, so it is still open.

In [5]:
robot = SemanticMiddleware(
    mode='resource', resource_iri=seed.MOBILE_ROBOT,
    service_class=seed.MOBILE_ROBOT_SERVICE_CLASS, ogm=OGM(db=GraphDB.from_env()),
    host='127.0.0.1', port=8998,
)

sparql = f'''
SELECT ?status_url ?open_url WHERE {{
    ?svc <{SVC.isServiceOf}> <{seed.DOOR_RESOURCE}> .
    ?sp <{SVC.isStatePropertyOf}> ?svc .
    ?sp a <{seed.DOOR_STATUS_STATE_CLASS}> .
    ?sp <{SVC.endpoint}> ?status_url .
    ?wf <{SVC.isWorkflowOf}> ?svc .
    ?wf a <{seed.DOOR_OPEN_WORKFLOW_CLASS}> .
    ?wf <{SVC.endpoint}> ?open_url .
}}'''
b = robot.ogm.db.query(sparql, convert_bindings=True)['results']['bindings'][0]
status_url, open_url = str(b['status_url']), str(b['open_url'])
print('discovered via SPARQL ->', status_url, '|', open_url)

def ensure_open(phase):
    state = httpx.get(status_url).json()
    print(f'  {phase}: door is {state}')
    if state == 'closed':
        httpx.post(open_url)  # invoke the open workflow directly at its execute URL
        state = httpx.get(status_url).json()
        print(f'  {phase}: opened it -> {state}')
    assert state == 'opened'

ensure_open('approach')
print('  drove through the door')
ensure_open('return')
print('  drove back through the door')

INFO:KafkaManager:KafkaManager initialized


INFO:GraphDB:Using GraphDB repository 'Tests' as user 'etienneh'.


INFO:httpx:HTTP Request: GET http://127.0.0.1:8997/state/door_status "HTTP/1.1 200 OK"


discovered via SPARQL -> http://127.0.0.1:8997/state/door_status | http://127.0.0.1:8997/workflows/door_open/execute
  approach: door is closed


INFO:httpx:HTTP Request: POST http://127.0.0.1:8997/workflows/door_open/execute "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://127.0.0.1:8997/state/door_status "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://127.0.0.1:8997/state/door_status "HTTP/1.1 200 OK"


  approach: opened it -> opened
  drove through the door
  return: door is opened
  drove back through the door


## Step 5 — The Live Status is Never Persisted

The status property carries only structural triples + its endpoint. The live `opened`/`closed` value is served from memory over REST and is never written to the graph.

In [6]:
for _, _, obj in db.triples_get(sub=status_sp):
    assert str(obj) not in ('opened', 'closed'), 'state value must not be persisted'
print("Confirmed: no 'opened'/'closed' literal on the state property.")

Confirmed: no 'opened'/'closed' literal on the state property.


## Step 6 — Shutdown and Deregistration

On shutdown the door removes its reachability triples (including the state property's endpoint) but preserves the individuals.

In [7]:
server.should_exit = True
thread.join(timeout=20)
time.sleep(0.5)
reset_door()
print('state endpoint removed:', not list(db.triples_get(sub=status_sp, pred=SVC.endpoint)))
print('state property preserved:', db.triple_exists((status_sp, RDF.type, seed.DOOR_STATUS_STATE_CLASS)))

state endpoint removed: True
state property preserved: True
